In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install pandas openpyxl

import pandas as pd

path = "/content/drive/MyDrive/mental_health_training_review_SAFETY_AUDIT.xlsx"

xls = pd.ExcelFile(path)
print(xls.sheet_names)

df = pd.read_excel(path, sheet_name="training_review")
print("Total records:", len(df))
print(df["decision"].value_counts(dropna=False))

['training_review', 'audit_summary']
Total records: 510
decision
revise      491
exclude      16
keep          2
escalate      1
Name: count, dtype: int64


In [3]:
fields = [
    "relevance",
    "empathy",
    "factual_grounding",
    "safety",
    "privacy",
    "decision"
]

for col in fields:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

print("\nMissing user inputs:", df["input"].isna().sum())
print("Missing assistant outputs:", df["output"].isna().sum())


relevance:
relevance
relevant             506
irrelevant             3
needs improvement      1
Name: count, dtype: int64

empathy:
empathy
appropriate          504
needs improvement      5
uncertain              1
Name: count, dtype: int64

factual_grounding:
factual_grounding
supported       265
questionable    203
unsupported      41
uncertain         1
Name: count, dtype: int64

safety:
safety
concerning     319
appropriate    188
unsafe           2
uncertain        1
Name: count, dtype: int64

privacy:
privacy
no concern    509
uncertain       1
Name: count, dtype: int64

decision:
decision
revise      491
exclude      16
keep          2
escalate      1
Name: count, dtype: int64

Missing user inputs: 2
Missing assistant outputs: 0


In [4]:
priority = df[
    (df["safety"].isin(["concerning", "unsafe", "uncertain"])) |
    (df["factual_grounding"].isin(["questionable", "unsupported", "uncertain"])) |
    (df["decision"].isin(["escalate", "exclude"])) |
    (df["input"].isna())
]

print("Records needing review:", len(priority))

display(priority[
    ["record_id", "input", "output", "factual_grounding",
     "safety", "decision", "review_notes"]
].head(10))

Records needing review: 473


,record_id,input,output,factual_grounding,safety,decision,review_notes
0,77,I've been attending ALS support group meetings...,I can empathize with the sense of isolation an...,questionable,appropriate,revise,Long and repetitive; introduces an unsupported...
1,78,(pausing to consider the counselor's words) I'...,It's understandable that you're feeling confli...,questionable,concerning,revise,Makes assumptions about the user's skills and ...
2,91,I've been struggling with my anxiety for years...,"Your anxiety is a complex issue, and it's unde...",questionable,appropriate,revise,Includes off-topic financial/family speculatio...
3,101,I've been struggling to find a way to deal wit...,I hear the pain and frustration in your voice ...,questionable,concerning,revise,The context around abuse is unclear; avoid ass...
5,110,I've been feeling so helpless and hopeless lat...,I'm truly sorry to hear that you've been feeli...,questionable,concerning,revise,Suicidal-thoughts response should address imme...
9,173,I've been feeling trapped in my house for week...,Your description of feeling trapped in your ho...,questionable,appropriate,revise,Contains unrelated business-planning material....
10,190,I've been struggling with feelings of anxiety ...,I'm sorry to hear that you've been experiencin...,supported,concerning,revise,Self-harm thoughts need direct safety-oriented...
11,193,"I feel like I'm trapped in this hospital bed, ...",I can see that you're feeling incredibly stuck...,unsupported,concerning,revise,Invents personal history such as childhood neg...
12,205,I've been so overwhelmed with taking care of m...,I can see how challenging it must be for you t...,questionable,concerning,revise,A family member's UTI needs appropriate medica...
13,215,I've been struggling with feelings of guilt an...,"I'm sorry for your loss, and I understand how ...",questionable,appropriate,revise,Grief response is long and generalizes about w...


In [5]:
row = df[df["record_id"] == 77].iloc[0]

print("USER INPUT:\n", row["input"])
print("\nASSISTANT RESPONSE:\n", row["output"])
print("\nCURRENT RATINGS:")
for col in [
    "relevance", "empathy", "factual_grounding",
    "safety", "privacy", "decision", "review_notes"
]:
    print(f"{col}: {row[col]}")

USER INPUT:
 I've been attending ALS support group meetings for six months now, hoping to meet someone in a similar situation as mine, but I haven't found anyone yet. Most attendees are middle-aged women taking care of their husbands, while I'm a younger man with a 39-year-old wife who has ALS and two young children to take care of. I keep feeling like there's always someone who has it worse, but I can't help but feel isolated in my situation.

ASSISTANT RESPONSE:
 I can empathize with the sense of isolation and frustration you're experiencing, as it's not uncommon for individuals in unique situations to feel disconnected from their support networks. However, it's essential to remember that everyone's experiences with ALS are different, and it's crucial to find ways to connect with others who can understand the complexities of your situation. The fact that most attendees are middle-aged women caring for their husbands doesn't diminish the significance of your experiences or the challen

In [6]:
import pandas as pd

input_path = "/content/drive/MyDrive/mental_health_training_review_SAFETY_AUDIT.xlsx"
output_path = "/content/drive/MyDrive/mental_health_training_review_UPDATED.xlsx"

df = pd.read_excel(input_path, sheet_name="training_review")

mask = df["record_id"] == 77

df.loc[mask, "output"] = (
    "It sounds lonely to attend support groups hoping to find someone who "
    "understands your situation, only to feel different from everyone else there. "
    "Caring for your wife while raising two young children brings challenges "
    "that may feel difficult to explain, and your struggles matter just as much "
    "as anyone else's. You don't have to compare your situation with others or "
    "feel guilty for needing support. You could ask the group facilitator whether "
    "they know of other younger spouses caring for someone with ALS, or whether "
    "an online support community might help you connect with people in similar "
    "circumstances. What do you wish other people in your support group understood "
    "about your experience?"
)

df.loc[mask, "relevance"] = "relevant"
df.loc[mask, "empathy"] = "appropriate"
df.loc[mask, "factual_grounding"] = "supported"
df.loc[mask, "safety"] = "appropriate"
df.loc[mask, "privacy"] = "no concern"
df.loc[mask, "decision"] = "revise"
df.loc[mask, "review_notes"] = (
    "Original response contains concatenated passages, invented counselling "
    "details, unsupported symptoms, and claims of an ongoing therapeutic "
    "relationship. Replaced with one grounded response."
)

assert mask.sum() == 1, f"Expected one record 77, found {mask.sum()}"

df.to_excel(output_path, index=False)

print("Updated record 77.")
print("Saved to:", output_path)

Updated record 77.
Saved to: /content/drive/MyDrive/mental_health_training_review_UPDATED.xlsx


In [7]:
row = df[df["record_id"] == 78].iloc[0]

print("USER INPUT:\n", row["input"])
print("\nASSISTANT RESPONSE:\n", row["output"])

USER INPUT:
 (pausing to consider the counselor's words) I've been thinking about it, and I believe the fear of going back to my old life is rooted in my past experiences. I've had a history of abuse, and the thought of returning to that environment is terrifying. But, at the same time, I feel trapped in the hospital. I've been here for months, and I'm tired of feeling like I'm stuck.

ASSISTANT RESPONSE:
 It's understandable that you're feeling conflicted. Your fear is a natural response to past traumas, and it's important to acknowledge and validate those feelings. However, it's also essential to remember that you've come a long way since then. You've gained valuable skills and insights during your time in the hospital, and you've built a strong support system. Let's explore some coping strategies that might help you manage your fears and prepare for the transition back to your old life.


In [8]:
df.loc[df["record_id"] == 78, "output"] = (
    "It makes sense that returning to an environment connected to past abuse "
    "could feel frightening, while staying in the hospital for months may also "
    "leave you feeling trapped. You deserve to feel safe, and you shouldn't "
    "have to return somewhere you fear being harmed. If you're comfortable, "
    "could you tell your care team about these fears so you can discuss a safe "
    "discharge plan and what support may be available? What worries you most "
    "about leaving the hospital?"
)

df.loc[df["record_id"] == 78, "relevance"] = "relevant"
df.loc[df["record_id"] == 78, "empathy"] = "appropriate"
df.loc[df["record_id"] == 78, "factual_grounding"] = "supported"
df.loc[df["record_id"] == 78, "safety"] = "appropriate"
df.loc[df["record_id"] == 78, "privacy"] = "no concern"
df.loc[df["record_id"] == 78, "decision"] = "revise"
df.loc[df["record_id"] == 78, "review_notes"] = (
    "Original assumes a strong support system and personal progress without "
    "evidence. Revised to validate fear, encourage discussion with the care "
    "team, and prioritize safe discharge planning."
)

df.to_excel(
    "/content/drive/MyDrive/mental_health_training_review_UPDATED.xlsx",
    index=False
)

print("Record 78 updated and saved.")

Record 78 updated and saved.


In [9]:
output_path = "/content/drive/MyDrive/mental_health_training_review_UPDATED.xlsx"

df.to_excel(output_path, index=False)

print("Saved:", output_path)

Saved: /content/drive/MyDrive/mental_health_training_review_UPDATED.xlsx


In [10]:
import pandas as pd
import json

# Save the latest review sheet
base = "/content/drive/MyDrive/"
df.to_excel(base + "mental_health_training_review_UPDATED.xlsx", index=False)

# Only approved 'keep' records; hold out all others
clean = df[
    (df["decision"] == "keep") &
    (df["input"].notna()) &
    (df["output"].notna()) &
    (df["input"].astype(str).str.strip() != "") &
    (df["output"].astype(str).str.strip() != "")
].copy()

# Extra safeguard: don't include records flagged as concerning/unsafe
clean = clean[
    clean["safety"].eq("appropriate") &
    clean["factual_grounding"].eq("supported") &
    clean["privacy"].eq("no concern") &
    clean["relevance"].eq("relevant") &
    clean["empathy"].eq("appropriate")
]

with open(base + "sft_draft.jsonl", "w", encoding="utf-8") as f:
    for _, r in clean.iterrows():
        f.write(json.dumps({
            "messages": [
                {"role": "user", "content": str(r["input"])},
                {"role": "assistant", "content": str(r["output"])}
            ]
        }, ensure_ascii=False) + "\n")

print("Spreadsheet saved.")
print("Draft SFT examples:", len(clean))
print("JSONL saved to:", base + "sft_draft.jsonl")
print("Do not train yet if the draft has too few examples.")

Spreadsheet saved.
Draft SFT examples: 0
JSONL saved to: /content/drive/MyDrive/sft_draft.jsonl
Do not train yet if the draft has too few examples.


In [11]:
import os

path = "/content/drive/MyDrive/sft_draft.jsonl"

print("File exists:", os.path.exists(path))
print("Training examples:", sum(1 for _ in open(path, encoding="utf-8")))

File exists: True
Training examples: 0


In [12]:
print(df["decision"].value_counts())

print("\nKeep records:")
print(df.loc[df["decision"] == "keep", [
    "record_id", "relevance", "empathy",
    "factual_grounding", "safety", "privacy"
]].to_string(index=False))

decision
revise      491
exclude      16
keep          2
escalate      1
Name: count, dtype: int64

Keep records:
 record_id relevance     empathy factual_grounding     safety    privacy
      1496  relevant appropriate       unsupported concerning no concern
      1501  relevant appropriate      questionable concerning no concern


In [13]:
for rid in [1496, 1501, 275]:
    row = df[df["record_id"] == rid]
    if not row.empty:
        print(f"\n--- Record {rid} ---")
        print("INPUT:", row.iloc[0]["input"])
        print("OUTPUT:", row.iloc[0]["output"])
        print("SAFETY:", row.iloc[0]["safety"])
        print("GROUNDING:", row.iloc[0]["factual_grounding"])
        print("DECISION:", row.iloc[0]["decision"])


--- Record 1496 ---
INPUT: I've been feeling overwhelmed with managing my aunt's care, especially when it comes to her eating habits. I've put up signs asking not to push the issue if she refuses, but I'm worried about signal fatigue and her English being a second language. I was hoping to discuss our goals during a recent meeting, but it didn't happen. Now that her condition has worsened, I believe it's essential to focus on recognizing and aligning our goals.
OUTPUT: Your concerns about your aunt's care, particularly her eating habits, are a reflection of your deep love and commitment to her well-being. It's understandable that you're feeling overwhelmed, especially given the language barrier and the potential for signal fatigue. I suggest focusing on clear, simple, and bold communication to minimize confusion and ensure that your messages are being received as intended. Additionally, having a conversation about your goals and hers could be a crucial step in this process. This discu

In [14]:
actions = {
    1496: ("exclude", "Concatenated unrelated cases; unsupported details."),
    1501: ("exclude", "Multiple unrelated caregiving responses concatenated."),
    275: ("escalate", "Suicidal intent mishandled; requires specialist safety review.")
}

for rid, (decision, note) in actions.items():
    mask = df["record_id"] == rid
    df.loc[mask, "decision"] = decision
    df.loc[mask, "review_notes"] = note

path = "/content/drive/MyDrive/mental_health_training_review_UPDATED.xlsx"
df.to_excel(path, index=False)

print("Updated and saved:", path)
print(df["decision"].value_counts())

Updated and saved: /content/drive/MyDrive/mental_health_training_review_UPDATED.xlsx
decision
revise      491
exclude      18
escalate      1
Name: count, dtype: int64


In [15]:
import pandas as pd
import os

path = "/content/drive/MyDrive/mental_health_training_review_UPDATED.xlsx"
df = pd.read_excel(path)

out_dir = "/content/drive/MyDrive/mental_health_review_outputs"
os.makedirs(out_dir, exist_ok=True)

# Separate records by decision
revise_df = df[df["decision"] == "revise"].copy()
exclude_df = df[df["decision"] == "exclude"].copy()
escalate_df = df[df["decision"] == "escalate"].copy()

revise_df.to_csv(f"{out_dir}/records_to_revise.csv", index=False)
exclude_df.to_csv(f"{out_dir}/records_excluded.csv", index=False)
escalate_df.to_csv(f"{out_dir}/records_for_specialist_review.csv", index=False)

# Full workbook with separate sheets
with pd.ExcelWriter(f"{out_dir}/review_queue.xlsx") as writer:
    df.to_excel(writer, sheet_name="All_Records", index=False)
    revise_df.to_excel(writer, sheet_name="Revise", index=False)
    exclude_df.to_excel(writer, sheet_name="Exclude", index=False)
    escalate_df.to_excel(writer, sheet_name="Escalate", index=False)

print("Files saved to:", out_dir)
print(f"Revise: {len(revise_df)} | Exclude: {len(exclude_df)} | Escalate: {len(escalate_df)}")

Files saved to: /content/drive/MyDrive/mental_health_review_outputs
Revise: 491 | Exclude: 18 | Escalate: 1


In [16]:
import pandas as pd
import os

path = "/content/drive/MyDrive/mental_health_training_review_UPDATED.xlsx"
df = pd.read_excel(path)

out_dir = "/content/drive/MyDrive/mental_health_review_outputs"
os.makedirs(out_dir, exist_ok=True)

# Separate records by decision
revise_df = df[df["decision"] == "revise"].copy()
exclude_df = df[df["decision"] == "exclude"].copy()
escalate_df = df[df["decision"] == "escalate"].copy()

revise_df.to_csv(f"{out_dir}/records_to_revise.csv", index=False)
exclude_df.to_csv(f"{out_dir}/records_excluded.csv", index=False)
escalate_df.to_csv(f"{out_dir}/records_for_specialist_review.csv", index=False)

# Full workbook with separate sheets
with pd.ExcelWriter(f"{out_dir}/review_queue.xlsx") as writer:
    df.to_excel(writer, sheet_name="All_Records", index=False)
    revise_df.to_excel(writer, sheet_name="Revise", index=False)
    exclude_df.to_excel(writer, sheet_name="Exclude", index=False)
    escalate_df.to_excel(writer, sheet_name="Escalate", index=False)

print("Files saved to:", out_dir)
print(f"Revise: {len(revise_df)} | Exclude: {len(exclude_df)} | Escalate: {len(escalate_df)}")

Files saved to: /content/drive/MyDrive/mental_health_review_outputs
Revise: 491 | Exclude: 18 | Escalate: 1


In [17]:
import pandas as pd
import re
import os

path = "/content/drive/MyDrive/mental_health_training_review_UPDATED.xlsx"
out = "/content/drive/MyDrive/mental_health_review_outputs"

df = pd.read_excel(path)
revise = df[df["decision"] == "revise"].copy()

def flag_output(value):
    text = str(value)
    flags = []

    if len(text) > 2500:
        flags.append("Very long output")
    if re.search(
        r"(ongoing counseling sessions|as we discussed in our previous session)",
        text, re.I
    ):
        flags.append("Possible fabricated therapy relationship")
    if len(re.findall(r"\n\s*(Assistant|Response|Therapist)\s*:", text, re.I)) > 1:
        flags.append("Possible joined responses")

    return "; ".join(flags)

revise["automated_flags"] = revise["output"].apply(flag_output)
flagged = revise[revise["automated_flags"] != ""]

os.makedirs(out, exist_ok=True)
flagged.to_csv(f"{out}/priority_output_review.csv", index=False)

print("Records scanned:", len(revise))
print("Flagged for closer review:", len(flagged))
print("Saved:", f"{out}/priority_output_review.csv")

Records scanned: 491
Flagged for closer review: 304
Saved: /content/drive/MyDrive/mental_health_review_outputs/priority_output_review.csv


In [18]:
import pandas as pd

source = "/content/drive/MyDrive/mental_health_review_outputs/priority_output_review.csv"
df = pd.read_csv(source)

df["output_length"] = df["output"].fillna("").astype(str).str.len()

priority = df.sort_values(
    by="output_length", ascending=False
).head(50)

target = "/content/drive/MyDrive/mental_health_review_outputs/top_50_priority.csv"
priority.to_csv(target, index=False)

print("Flagged records:", len(df))
print("Top-priority records saved:", len(priority))
print("File:", target)
display(priority[
    ["record_id", "output_length", "automated_flags", "decision"]
].head(15))

Flagged records: 304
Top-priority records saved: 50
File: /content/drive/MyDrive/mental_health_review_outputs/top_50_priority.csv


,record_id,output_length,automated_flags,decision
32,1858,12275,Very long output,revise
282,12213,10508,Very long output,revise
92,4172,9177,Very long output,revise
106,4466,9111,Very long output,revise
124,4837,8192,Very long output,revise
121,4802,8192,Very long output,revise
28,1794,8105,Very long output,revise
299,12632,7955,Very long output,revise
123,4805,7805,Very long output,revise
94,4192,7641,Very long output,revise


In [19]:
import pandas as pd
import re
import os

base = "/content/drive/MyDrive/mental_health_review_outputs"
df = pd.read_csv(f"{base}/top_50_priority.csv")

def inspect_output(text):
    text = str(text)
    flags = []

    # Repeated conversational section markers
    markers = re.findall(
        r"(?im)^\s*(?:user|assistant|therapist|response)\s*:",
        text
    )
    if len(markers) >= 3:
        flags.append(f"Multiple section markers ({len(markers)})")

    # Repeated paragraphs may indicate accidental duplication
    paragraphs = [
        p.strip() for p in re.split(r"\n\s*\n", text)
        if len(p.strip()) > 100
    ]
    if len(paragraphs) != len(set(paragraphs)):
        flags.append("Repeated paragraphs")

    # Common signs of an appended unrelated conversation
    if re.search(
        r"(?i)(in our previous session|as your therapist|"
        r"our ongoing counseling|as we discussed last time)",
        text
    ):
        flags.append("Possible unsupported therapy relationship")

    if len(text) > 2500:
        flags.append("Manual content check needed: long output")

    return "; ".join(flags)

df["contamination_check"] = df["output"].apply(inspect_output)

df.to_csv(f"{base}/top_50_contamination_check.csv", index=False)

print(df["contamination_check"].value_counts().to_string())
print("Saved:", f"{base}/top_50_contamination_check.csv")

display(df[
    ["record_id", "output_length", "contamination_check"]
].head(20))

contamination_check
Manual content check needed: long output                                                                    44
Repeated paragraphs; Manual content check needed: long output                                                4
Repeated paragraphs; Possible unsupported therapy relationship; Manual content check needed: long output     1
Possible unsupported therapy relationship; Manual content check needed: long output                          1
Saved: /content/drive/MyDrive/mental_health_review_outputs/top_50_contamination_check.csv


,record_id,output_length,contamination_check
0,1858,12275,Repeated paragraphs; Possible unsupported ther...
1,12213,10508,Manual content check needed: long output
2,4172,9177,Manual content check needed: long output
3,4466,9111,Manual content check needed: long output
4,4837,8192,Repeated paragraphs; Manual content check need...
5,4802,8192,Repeated paragraphs; Manual content check need...
6,1794,8105,Manual content check needed: long output
7,12632,7955,Manual content check needed: long output
8,4805,7805,Manual content check needed: long output
9,4192,7641,Manual content check needed: long output


In [21]:
import pandas as pd
import re
import os

base = "/content/drive/MyDrive/mental_health_review_outputs"
df = pd.read_csv(f"{base}/top_50_contamination_check.csv")

def remove_exact_duplicate_paragraphs(text):
    text = str(text)
    paragraphs = re.split(r"\n\s*\n", text.strip())
    seen = set()
    kept = []

    for p in paragraphs:
        normalized = re.sub(r"\s+", " ", p).strip().casefold()
        if normalized and normalized not in seen:
            seen.add(normalized)
            kept.append(p.strip())

    return "\n\n".join(kept)

df["output_original"] = df["output"]
df["output_deduplicated_candidate"] = df["output"].apply(
    remove_exact_duplicate_paragraphs
)
df["changed"] = (
    df["output_original"] != df["output_deduplicated_candidate"]
)

out_path = f"{base}/top_50_deduplication_candidates.csv"
df.to_csv(out_path, index=False)

print("Records examined:", len(df))
print("Records with exact duplicate paragraphs:", int(df["changed"].sum()))
print("Saved comparison file:", out_path)
print("IMPORTANT: Candidate text is not automatically approved for training.")

Records examined: 50
Records with exact duplicate paragraphs: 10
Saved comparison file: /content/drive/MyDrive/mental_health_review_outputs/top_50_deduplication_candidates.csv
IMPORTANT: Candidate text is not automatically approved for training.


In [22]:
import pandas as pd

base = "/content/drive/MyDrive/mental_health_review_outputs"

df = pd.read_csv(f"{base}/top_50_deduplication_candidates.csv")
changed = df[df["changed"] == True].copy()

changed.to_csv(
    f"{base}/10_duplicate_paragraph_candidates.csv",
    index=False
)

print("Records needing duplicate-paragraph review:", len(changed))
display(changed[[
    "record_id",
    "output_original",
    "output_deduplicated_candidate"
]].head(10))

Records needing duplicate-paragraph review: 10


,record_id,output_original,output_deduplicated_candidate
0,1858,"Your feelings are understandable, given the co...","Your feelings are understandable, given the co..."
4,4837,"Your situation is undoubtedly challenging, and...","Your situation is undoubtedly challenging, and..."
5,4802,Great question! Cognitive-behavioral technique...,Great question! Cognitive-behavioral technique...
13,2343,I'm sorry for the difficulties you've faced in...,I'm sorry for the difficulties you've faced in...
26,12218,"Behavioral change can indeed be challenging, b...","Behavioral change can indeed be challenging, b..."
30,3376,Your sleep disturbances seem to be linked to y...,Your sleep disturbances seem to be linked to y...
37,11671,"First, I want to acknowledge your bravery in r...","First, I want to acknowledge your bravery in r..."
40,4940,Your description of feeling emotionally and ph...,Your description of feeling emotionally and ph...
42,226,I understand that you're in a challenging situ...,I understand that you're in a challenging situ...
46,1777,I'm here to listen and support you during this...,I'm here to listen and support you during this...


In [23]:
import pandas as pd
import json, os, re

src = "/content/drive/MyDrive/mental_health_training_review_UPDATED.xlsx"
out = "/content/drive/MyDrive/mental_health_review_outputs"
os.makedirs(out, exist_ok=True)

df = pd.read_excel(src)

# Normalize labels and text
for col in ["decision", "relevance", "empathy",
            "factual_grounding", "safety", "privacy"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()

df["input"] = df["input"].fillna("").astype(str)
df["output"] = df["output"].fillna("").astype(str)
df["output_length"] = df["output"].str.len()

# Conservative screening: candidates only, not clinical approval
risk_pattern = re.compile(
    r"\b(suicid\w*|kill myself|end my life|self[- ]?harm|"
    r"overdos\w*|want to die|abuse|psychos\w*)\b", re.I
)

df["risk_keyword_flag"] = (
    df["input"].str.contains(risk_pattern, na=False) |
    df["output"].str.contains(risk_pattern, na=False)
)

df["duplicate_paragraph_flag"] = df["output"].apply(
    lambda s: len([
        p.strip().casefold()
        for p in re.split(r"\n\s*\n", s.strip()) if p.strip()
    ]) != len(set([
        p.strip().casefold()
        for p in re.split(r"\n\s*\n", s.strip()) if p.strip()
    ]))
)

eligible = (
    df["decision"].eq("keep") &
    df["relevance"].eq("relevant") &
    df["empathy"].eq("appropriate") &
    df["factual_grounding"].eq("supported") &
    df["safety"].eq("appropriate") &
    df["privacy"].eq("no concern") &
    df["input"].str.strip().ne("") &
    df["output"].str.strip().ne("") &
    df["output_length"].le(2500) &
    ~df["risk_keyword_flag"] &
    ~df["duplicate_paragraph_flag"]
)

candidates = df[eligible].copy()
review_queue = df[
    df["decision"].eq("revise") |
    df["risk_keyword_flag"] |
    df["duplicate_paragraph_flag"] |
    df["output_length"].gt(2500)
].copy()

# Export provisional JSONL only from strict candidates
jsonl_path = f"{out}/sft_provisional.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for _, row in candidates.iterrows():
        item = {
            "messages": [
                {"role": "user", "content": row["input"]},
                {"role": "assistant", "content": row["output"]}
            ],
            "record_id": str(row.get("record_id", ""))
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

review_queue.to_csv(f"{out}/final_review_queue.csv", index=False)
df.to_excel(f"{out}/final_audit_workbook.xlsx", index=False)

report = {
    "total_records": len(df),
    "provisional_sft_candidates": len(candidates),
    "review_queue_records": len(review_queue),
    "excluded": int(df["decision"].eq("exclude").sum()),
    "escalated": int(df["decision"].eq("escalate").sum()),
    "risk_keyword_flags": int(df["risk_keyword_flag"].sum()),
    "duplicate_paragraph_flags": int(df["duplicate_paragraph_flag"].sum()),
    "long_outputs_over_2500_chars": int(df["output_length"].gt(2500).sum()),
    "note": "Automated screening only; not clinical validation."
}

with open(f"{out}/submission_audit_summary.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print("\nFiles saved in:", out)

/tmp/ipykernel_502/784697571.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["input"].str.contains(risk_pattern, na=False) |
/tmp/ipykernel_502/784697571.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["output"].str.contains(risk_pattern, na=False)


{
  "total_records": 510,
  "provisional_sft_candidates": 0,
  "review_queue_records": 509,
  "excluded": 18,
  "escalated": 1,
  "risk_keyword_flags": 341,
  "duplicate_paragraph_flags": 5,
  "long_outputs_over_2500_chars": 321,
  "note": "Automated screening only; not clinical validation."
}

Files saved in: /content/drive/MyDrive/mental_health_review_outputs


In [24]:
import pandas as pd
import os

src = "/content/drive/MyDrive/mental_health_training_review_UPDATED.xlsx"
out = "/content/drive/MyDrive/mental_health_review_outputs"

df = pd.read_excel(src)

cols = [
    "decision", "relevance", "empathy",
    "factual_grounding", "safety", "privacy"
]

for c in cols:
    df[c] = df[c].fillna("missing").astype(str).str.strip().str.lower()

print("=== QUALITY LABEL COUNTS ===")
for c in cols:
    print(f"\n{c.upper()}")
    print(df[c].value_counts().to_string())

print("\n=== DECISION × SAFETY ===")
print(pd.crosstab(df["decision"], df["safety"]).to_string())

print("\n=== POTENTIAL REVISION POOL ===")
pool = df[
    df["decision"].eq("revise") &
    df["input"].fillna("").astype(str).str.strip().ne("") &
    df["output"].fillna("").astype(str).str.strip().ne("")
].copy()

pool.to_csv(f"{out}/revision_pool.csv", index=False)
print("Non-empty revise records:", len(pool))
print("Saved:", f"{out}/revision_pool.csv")

=== QUALITY LABEL COUNTS ===

DECISION
decision
revise      491
exclude      18
escalate      1

RELEVANCE
relevance
relevant             506
irrelevant             3
needs improvement      1

EMPATHY
empathy
appropriate          504
needs improvement      5
uncertain              1

FACTUAL_GROUNDING
factual_grounding
supported       267
questionable    201
unsupported      41
uncertain         1

SAFETY
safety
concerning     318
appropriate    189
unsafe           2
uncertain        1

PRIVACY
privacy
no concern    509
uncertain       1

=== DECISION × SAFETY ===
safety    appropriate  concerning  uncertain  unsafe
decision                                            
escalate            0           0          0       1
exclude            10           6          1       1
revise            179         312          0       0

=== POTENTIAL REVISION POOL ===
Non-empty revise records: 491
Saved: /content/drive/MyDrive/mental_health_review_outputs/revision_pool.csv


In [25]:
import pandas as pd
import os

base = "/content/drive/MyDrive/mental_health_review_outputs"
df = pd.read_excel(
    "/content/drive/MyDrive/mental_health_training_review_UPDATED.xlsx"
)

os.makedirs(base, exist_ok=True)

columns = [
    "decision", "relevance", "empathy",
    "factual_grounding", "safety", "privacy"
]

summary = []
for col in columns:
    counts = df[col].fillna("missing").astype(str).str.strip().value_counts()
    for label, count in counts.items():
        summary.append({
            "quality_dimension": col,
            "label": label,
            "count": int(count),
            "percentage": round(count / len(df) * 100, 2)
        })

summary_df = pd.DataFrame(summary)
summary_df.to_csv(f"{base}/quality_summary.csv", index=False)

report = f"""# Mental-Health Conversational Dataset: Safety Audit

## Dataset overview
- Total records: {len(df)}
- Records marked revise: {(df['decision'] == 'revise').sum()}
- Records marked exclude: {(df['decision'] == 'exclude').sum()}
- Records marked escalate: {(df['decision'] == 'escalate').sum()}

## Audit findings
{summary_df.to_markdown(index=False)}

## Interpretation
Automated and preliminary review labels identify records requiring further
inspection. They are not a substitute for clinical validation.

The strict initial SFT filter produced zero eligible examples. Therefore,
no records are currently certified as suitable for training. Records marked
revise must be corrected and checked before inclusion. Excluded records must
remain out of the training set, and escalated high-risk examples require
specialist review.

## Limitations
- Keyword and structural checks can produce false positives and false negatives.
- Label quality depends on the preliminary review process.
- No clinical safety certification or model-performance claim is made.
- A final training dataset requires validated responses and documented quality
  checks.

## Next steps
1. Remove or rewrite concatenated and unsupported responses.
2. Review high-risk mental-health examples with qualified oversight.
3. Validate revised responses against a documented rubric.
4. Export only approved examples into the model's required chat format.
5. Evaluate the trained model on a separate safety and quality test set.
"""

with open(f"{base}/audit_report.md", "w", encoding="utf-8") as f:
    f.write(report)

print("Created:")
print(f"{base}/quality_summary.csv")
print(f"{base}/audit_report.md")
print("\nYou can now use these as submission supporting files.")


Created:
/content/drive/MyDrive/mental_health_review_outputs/quality_summary.csv
/content/drive/MyDrive/mental_health_review_outputs/audit_report.md

You can now use these as submission supporting files.
